# 🧹 Module 3 — Class 1: Telco Customer Churn — Cleaning Lab

**Dataset:** Telco Customer Churn (real telecom data, ~7,000 customers)

**Lecture:** [https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

## 👋 Read this first — what is this notebook?

A **notebook** is a document that mixes:
- 📝 **Markdown cells** (text like this) — they explain ideas.
- 💻 **Code cells** — Python you can run. The output appears below the cell.

**To run a code cell:** click on it, then press **Shift + Enter**.

## 🌐 If this is your first time using Google Colab

**Colab** is Google's free online Python notebook. You don't install anything — it runs in your browser.

1. Open this notebook in Colab (File → Open Notebook → GitHub tab).
2. Click **Runtime → Run all** to run every cell, OR
3. Run cells one by one with **Shift + Enter** (recommended for learning).

## 📦 IMPORTANT — read every comment

Every code cell has **comments** — lines that start with `#`. **Comments are notes for humans. Python ignores them.** They tell you what each line does. **DO NOT SKIP THEM.** They are the most important part of this notebook for first-time learners.

Example:
```python
x = 5      # 'x = 5' creates a variable called x and stores 5 inside it.
print(x)   # 'print()' shows a value on screen. This will display 5.
```

## 🎯 What you will practice today

- **Inspecting** real-world data structure and types
- **Detecting and fixing** type conversion issues (numbers stored as strings)
- **Handling missing values** — mean / median / mode imputation, missing indicators
- **Standardizing** inconsistent categorical entries
- **Detecting outliers** with the IQR method and Z-scores

## 🧠 The Golden Rule

> **Never delete data silently.** Log what you dropped and why.

Bad data fed to a strong model = confidently wrong predictions. That is worse than no model. Data scientists spend ~80% of their time cleaning data — so do it well.

Let's go. 🚀

---

## 0. 🔧 Setup — Import the Tools We Need

Before we touch any data, we **import** the libraries (other people's reusable code) that pandas, numpy, etc. provide. Read the comments in the next cell carefully — they teach you what each library does and why we rename them.

In [ ]:
# ────────────────────────────────────────────────────────────
# 'import' brings in code that someone else already wrote.
# 'as' gives the library a short nickname so we type less.
# These four libraries are the standard data-science toolkit.
# ────────────────────────────────────────────────────────────

import pandas as pd               # pandas → tables (rows + columns). Nickname: pd
import numpy as np                # numpy  → numerical helpers (math, arrays). Nickname: np
import matplotlib.pyplot as plt   # matplotlib.pyplot → drawing charts. Nickname: plt
import seaborn as sns             # seaborn → prettier statistical charts on top of matplotlib. Nickname: sns

# ────────────────────────────────────────────────────────────
# 'warnings' is a built-in Python module (no install needed).
# We silence non-critical warnings so the output stays clean.
# In real production code you'd usually leave warnings ON.
# ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────────────────
# print() displays text on screen.
# Anything in quotes is a 'string' (text). Strings are how we
# label things and write messages.
# ────────────────────────────────────────────────────────────
print("Setup complete. All four libraries imported.")

---
## 1. 📥 Load the Data

We'll load the Telco Customer Churn dataset directly from the internet. **No download needed** — pandas can read CSV files straight from a URL.

### 📚 What is a CSV?

A **CSV** ("comma-separated values") is a plain text file where each line is a row, and columns are separated by commas. Excel and Google Sheets can save as CSV. It's the most common way to share tabular data.

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 1: Save the URL of the CSV file in a variable.
# A 'variable' is a labelled container for a value.
# Here, 'url' is the variable name; everything in quotes is the value.
# ────────────────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# ────────────────────────────────────────────────────────────
# STEP 2: Tell pandas to read that CSV.
# pd.read_csv(url) goes to the URL, downloads the file, and turns
# it into a DataFrame (a table). We store the table in 'df'.
# 'df' is just a common short name for 'DataFrame'.
# ────────────────────────────────────────────────────────────
df = pd.read_csv(url)

# ────────────────────────────────────────────────────────────
# STEP 3: Print how many rows and columns we got.
# df.shape returns a 'tuple' (rows, columns) — e.g. (7043, 21).
# df.shape[0] = first item = rows. df.shape[1] = second item = columns.
#
# The 'f' before the string makes it an 'f-string'.
# Inside { } you can put any Python expression — it gets inserted into the text.
# ────────────────────────────────────────────────────────────
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

# ────────────────────────────────────────────────────────────
# STEP 4: Peek at the first 5 rows.
# .head() returns the top 5 rows of a DataFrame by default.
# This is ALWAYS the first thing you do after loading data.
# ────────────────────────────────────────────────────────────
df.head()

In [ ]:
# ────────────────────────────────────────────────────────────
# FALLBACK: only run THIS cell if the URL above fails.
# Lines starting with '#' are comments — to USE the code below,
# you'd remove the leading '#' (this is called 'uncommenting').
#
# This block uses Colab's file-upload widget so you can upload
# Telco-Customer-Churn.csv from your laptop instead of the URL.
# ────────────────────────────────────────────────────────────

# from google.colab import files                 # Colab's upload helper
# uploaded = files.upload()                       # opens a file picker
# filename = list(uploaded.keys())[0]             # grabs the file's name
# df = pd.read_csv(filename)                      # load it as a DataFrame
# print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

---
## 2. 🔍 Inspect the Structure — Look Before You Touch

Before changing a single value, we audit the data. Five quick commands tell us everything we need to know.

### 📚 The five inspection commands

| Command | What it shows |
| --- | --- |
| `df.shape` | `(rows, columns)` — size at a glance |
| `df.info()` | dtypes + non-null counts per column |
| `df.describe()` | summary stats (mean, std, min, max...) for numeric columns |
| `df.describe(include='object')` | summary for **text** columns |
| `df.isnull().sum()` | count of missing values per column |

In [ ]:
# ────────────────────────────────────────────────────────────
# .shape returns a TUPLE of (rows, cols).
# Notice there are NO PARENTHESES after .shape — it's an attribute,
# not a function call. (Compare to .info() below, which is a function.)
# ────────────────────────────────────────────────────────────
print("Shape:", df.shape)
print()  # empty print() just adds a blank line for readability

In [ ]:
# ────────────────────────────────────────────────────────────
# .info() prints column names, dtypes, and non-null counts.
# It's a FUNCTION, so we add () at the end to actually run it.
#
# Look at the 'Dtype' column in the output:
#   'object'  = text (string)
#   'int64'   = whole number (integer)
#   'float64' = decimal number
#
# Look at 'Non-Null Count': if it's less than the total rows,
# that column has missing values.
# ────────────────────────────────────────────────────────────
df.info()

In [ ]:
# ────────────────────────────────────────────────────────────
# .describe() shows summary statistics for NUMERIC columns only:
#   count = how many non-missing values
#   mean  = average
#   std   = standard deviation (how spread out)
#   min   = smallest value
#   25%   = first quartile (25% of values are below this)
#   50%   = median (the middle value)
#   75%   = third quartile
#   max   = largest value
#
# Always check min / max — extremes spot bad data fast.
# ────────────────────────────────────────────────────────────
df.describe()

In [ ]:
# ────────────────────────────────────────────────────────────
# Same as above but for TEXT columns. include='object' tells pandas:
#   "summarize columns whose dtype is 'object' (i.e. text)".
#
# What you'll see:
#   count  = total non-null values
#   unique = number of distinct values
#   top    = most common value
#   freq   = how many times the most common value appears
# ────────────────────────────────────────────────────────────
df.describe(include='object')

### ✅ Check Your Understanding

Look at the four outputs above and answer:

1. How many rows and columns are in the dataset?
2. Look at `df.info()` — which column is **`object`** but probably *should* be numeric?
3. Look at `df.describe()` — what is the **maximum** `tenure`? What unit is it in (months, years)?
4. Look at `df.describe(include='object')` — which column has the **most unique values**? Why?

---
## 3. 🛠️ Fix the `TotalCharges` Type Issue

**The problem.** `TotalCharges` should be a number (dollars), but pandas read it as `object` (text). That means we can't compute means, medians, or do math on it. We need to fix this **before** anything else.

**The reason** (you'll discover this in a second): some rows have a **blank string** instead of a number — that breaks pandas' auto-type-detection.

### 📚 The safe conversion pattern

```python
df['col'] = pd.to_numeric(df['col'], errors='coerce')
```

- `pd.to_numeric` tries to turn each value into a number.
- `errors='coerce'` — when a value **cannot** be converted (like a blank string), turn it into `NaN` instead of crashing.
- This is the **safest, most common** numeric-conversion pattern in pandas.

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 1: Confirm the broken dtype.
# df['TotalCharges'] selects ONE column (returns a Series — like a single-column DataFrame).
# .dtype shows that Series's data type.
# ────────────────────────────────────────────────────────────
print("TotalCharges dtype:", df['TotalCharges'].dtype)

# ────────────────────────────────────────────────────────────
# STEP 2: Find the rows that fail to convert.
# pd.to_numeric(..., errors='coerce') turns bad values into NaN.
# .isna() returns True/False per row — True wherever the value is NaN.
# We store that True/False mask in 'mask' so we can use it twice.
# ────────────────────────────────────────────────────────────
mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()

# ────────────────────────────────────────────────────────────
# .sum() on a True/False Series counts the Trues (True = 1, False = 0).
# So mask.sum() = how many rows can't be converted to a number.
# ────────────────────────────────────────────────────────────
print(f"\nRows that can't be converted to numeric: {mask.sum()}")

# ────────────────────────────────────────────────────────────
# STEP 3: Look at the bad rows so we understand WHY they fail.
# df.loc[mask, [...]] reads as: 'rows where mask is True, only these columns'.
# Notice: many of these customers have tenure = 0 (brand new customers).
# ────────────────────────────────────────────────────────────
print("\nProblematic values:")
df.loc[mask, ['customerID', 'TotalCharges', 'tenure', 'MonthlyCharges']]

In [ ]:
# ────────────────────────────────────────────────────────────
# Now actually convert the column.
# This OVERWRITES df['TotalCharges'] with the numeric version.
# Every blank-string row becomes NaN automatically (thanks to coerce).
# ────────────────────────────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Confirm the fix worked
print("TotalCharges dtype after fix:", df['TotalCharges'].dtype)
print(f"NaN count in TotalCharges: {df['TotalCharges'].isna().sum()}")

In [ ]:
# ────────────────────────────────────────────────────────────
# Domain reasoning: a customer with tenure=0 is a brand-new
# customer who has never been billed yet. So their TotalCharges
# is logically 0, not 'unknown'. We can safely fill those NaNs with 0.
#
# This is a perfect example of WHY domain knowledge matters:
# blindly imputing the median would be WRONG here. Always think
# about what the missingness MEANS before filling.
# ────────────────────────────────────────────────────────────
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"NaN count after fill: {df['TotalCharges'].isna().sum()}")

### ✅ Check Your Understanding

1. **Why** did pandas read `TotalCharges` as text? What was inside the bad rows?
2. We chose `fillna(0)` for the `TotalCharges` NaNs. **Why is filling with 0 correct here**, when in most other cases filling with 0 is dangerous?
3. What would have happened if we used `errors='raise'` (the default) instead of `'coerce'`? Don't run it — predict.

---

## 4. 🕳️ Missing Values

### 4a. Check for missing values across all columns

In [ ]:
# ────────────────────────────────────────────────────────────
# .isnull() and .isna() are identical — both return a True/False table
# where True = the value is missing.
# .sum() then counts Trues per column (sum across rows).
# ────────────────────────────────────────────────────────────
missing = df.isnull().sum()

# ────────────────────────────────────────────────────────────
# Convert raw counts into percentages — usually more useful.
# Division of a Series by a number divides each element.
# .round(2) keeps just 2 decimal places (cleaner output).
# ────────────────────────────────────────────────────────────
missing_pct = (missing / len(df) * 100).round(2)

# ────────────────────────────────────────────────────────────
# Build a small DataFrame to display count + percent side by side.
# pd.DataFrame({'colname': series, ...}) makes a new table.
# ────────────────────────────────────────────────────────────
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})

# ────────────────────────────────────────────────────────────
# Filter to only show columns that ACTUALLY have missing values.
# missing_df['count'] > 0 returns True/False per row.
# missing_df[ ... ] keeps only the True rows.
# ────────────────────────────────────────────────────────────
missing_df[missing_df['count'] > 0]

### 4b. Practice imputation methods

**Imputation** = filling missing values with a guess. The Telco data is now clean, so to *practice*, we'll deliberately inject 200 fake missing values into `MonthlyCharges` and try three filling strategies.

### 📚 Three classic imputation strategies

| Strategy | When to use it |
| --- | --- |
| **Mean** (average) | Clean numeric data, no outliers |
| **Median** (middle value) | Numeric data, especially with outliers (more robust) |
| **Mode** (most common) | Categorical data (text columns) |

**Why median > mean?** A single outlier (like one customer paying $99,999) drags the mean far off. The median ignores outliers — half the values are below, half above.

In [ ]:
# ────────────────────────────────────────────────────────────
# np.random.seed(42) — make the randomness reproducible.
# Without a seed, every run picks different rows. With seed=42,
# everyone running this notebook gets THE SAME random rows.
# ────────────────────────────────────────────────────────────
np.random.seed(42)

# ────────────────────────────────────────────────────────────
# .copy() makes an INDEPENDENT copy of df. Without .copy(), changing
# df_demo would also change df — a classic pandas footgun.
# ────────────────────────────────────────────────────────────
df_demo = df.copy()

# ────────────────────────────────────────────────────────────
# Pick 200 random row indices (without repeats).
#   df_demo.index = the row labels (0, 1, 2, ..., 7042)
#   size=200 = pick 200
#   replace=False = no duplicates
# ────────────────────────────────────────────────────────────
nan_idx = np.random.choice(df_demo.index, size=200, replace=False)

# ────────────────────────────────────────────────────────────
# Use df.loc[ row_labels, column_name ] to assign NaN.
# np.nan is the official 'missing value' marker in pandas.
# ────────────────────────────────────────────────────────────
df_demo.loc[nan_idx, 'MonthlyCharges'] = np.nan

# Confirm we injected 200 NaNs
print(f"Injected {df_demo['MonthlyCharges'].isna().sum()} NaN values into MonthlyCharges")

In [ ]:
# ────────────────────────────────────────────────────────────
# STRATEGY 1: MEAN imputation.
# Compute the mean of the non-NaN values, then fill all NaN with it.
# .mean() automatically skips NaN, so this is safe even with missing data.
# ────────────────────────────────────────────────────────────
mean_val = df_demo['MonthlyCharges'].mean()

# Make a fresh copy to keep df_demo unchanged for the next strategy.
df_mean = df_demo.copy()

# .fillna(value) replaces every NaN in the column with `value`.
df_mean['MonthlyCharges'] = df_mean['MonthlyCharges'].fillna(mean_val)

# {mean_val:.2f} — format the number with exactly 2 decimal places.
print(f"Mean imputation value: {mean_val:.2f}")
print(f"Remaining NaN: {df_mean['MonthlyCharges'].isna().sum()}")

In [ ]:
# ────────────────────────────────────────────────────────────
# STRATEGY 2: MEDIAN imputation.
# Same idea but uses the MEDIAN (middle value, robust to outliers).
# ────────────────────────────────────────────────────────────
median_val = df_demo['MonthlyCharges'].median()

df_median = df_demo.copy()
df_median['MonthlyCharges'] = df_median['MonthlyCharges'].fillna(median_val)

print(f"Median imputation value: {median_val:.2f}")
print(f"Remaining NaN: {df_median['MonthlyCharges'].isna().sum()}")

# ────────────────────────────────────────────────────────────
# Compare the two — usually mean and median are close but not equal.
# A big gap means the data is SKEWED (a few extreme values pulling the mean).
# ────────────────────────────────────────────────────────────
print(f"\nDifference (mean - median): {mean_val - median_val:.2f}")

### 4c. 📝 TODO — Mode imputation + Missing Indicator

Now you finish the third strategy.

**TODO 1: Mode imputation.** Mode = the most common value. For numeric columns it's rare to use it, but it's the GO-TO for categorical (text) columns. Pandas returns it as a Series, so you grab the first element with `[0]`.

**TODO 2: Missing indicator.** A common trick: BEFORE you fill NaNs, create a *new column* that records which rows used to be missing. The model can sometimes learn that the *fact of missingness* itself carries information.

### 📚 Hints

```python
# Mode imputation
mode_val = df_demo['MonthlyCharges'].mode()[0]   # mode() returns a Series; [0] grabs the first
df_mode['MonthlyCharges'] = df_mode['MonthlyCharges'].fillna(mode_val)

# Missing indicator (do this BEFORE the fillna!)
df_mode['MonthlyCharges_missing'] = df_mode['MonthlyCharges'].isna().astype(int)
# .astype(int) converts True/False to 1/0
```

In [ ]:
# ────────────────────────────────────────────────────────────
# TODO 1: Perform mode imputation on df_demo['MonthlyCharges']
# Hint: use df_demo['MonthlyCharges'].mode()[0]
# Store the result in df_mode
# ────────────────────────────────────────────────────────────
df_mode = df_demo.copy()

# YOUR CODE HERE — compute mode_val and fillna with it


# ────────────────────────────────────────────────────────────
# TODO 2: Create a binary missing-indicator column called
# 'MonthlyCharges_missing'. It should be 1 where MonthlyCharges
# was NaN, 0 otherwise.
#
# IMPORTANT: add the indicator BEFORE you fillna. After fillna,
# .isna() would return all False — too late.
# ────────────────────────────────────────────────────────────

# YOUR CODE HERE — add the missing indicator


# Verification (uncomment after writing your code):
# print(f"Mode value: {mode_val:.2f}")
# print(f"Missing indicator distribution:\n{df_mode['MonthlyCharges_missing'].value_counts()}")

### ✅ Check Your Understanding

1. Compare your mode value to the mean and median you got earlier. Are they close? Why or why not? (Hint: `MonthlyCharges` has many distinct values — maybe none are repeated often.)
2. Think of a real scenario where the **fact that a value is missing** is useful information. Example: a customer who didn't fill in `referral_code` may have signed up directly, not through a friend.
3. **Order of operations.** Why MUST you create the missing indicator BEFORE running `.fillna()`?

---

## 5. 🧽 Inconsistent Categorical Entries

Now we look at all the **text** columns. Real data often has multiple spellings of the same thing ("Yes", "yes", "YES", " Yes "). Models would treat those as four different categories. We need exactly one spelling per category.

In [ ]:
# ────────────────────────────────────────────────────────────
# .select_dtypes(include='object') keeps ONLY the text columns.
# .columns gives the column names.
# .tolist() converts to a regular Python list (easier to manipulate).
# ────────────────────────────────────────────────────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Remove customerID — it's a unique ID per row, not a real category.
cat_cols.remove('customerID')

print(f"Categorical columns to inspect: {len(cat_cols)}")
print(cat_cols)

In [ ]:
# ────────────────────────────────────────────────────────────
# Loop through each categorical column and print its value counts.
# A 'for' loop iterates over each item in a list:
#   for col in cat_cols:
#       <do something with col>
# ────────────────────────────────────────────────────────────
for col in cat_cols:
    # .value_counts() returns a Series: each unique value -> how many times it appears
    vc = df[col].value_counts()

    # df[col].nunique() = number of distinct values in this column
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(vc)
    print()  # blank line between columns

In [ ]:
# ────────────────────────────────────────────────────────────
# Look at the output above. Notice that several columns have these
# verbose values:
#   'No internet service'  — for internet-feature columns
#   'No phone service'     — for phone-feature columns
#
# Functionally these mean 'No' — the customer doesn't have the feature.
# Treating them as separate values inflates cardinality without
# adding information. Standardize them down to plain 'No'.
#
# Below: programmatically find which columns have these patterns.
# ────────────────────────────────────────────────────────────
for col in cat_cols:
    vals = df[col].unique()                              # all distinct values in this column

    # List comprehension: keep only values that contain 'No ' AND aren't just 'No'.
    # str(v) ensures we can call 'in' even if v is not a string (defensive).
    special = [v for v in vals if 'No ' in str(v) and v != 'No']

    # Only print columns that have these special values
    if special:
        print(f"{col}: {special}")

In [ ]:
# ────────────────────────────────────────────────────────────
# Replace 'No internet service' with 'No' in the columns where it appears.
# .replace(old, new) on a Series swaps every match.
# ────────────────────────────────────────────────────────────
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# Loop through each column and apply the replacement
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')

# 'MultipleLines' uses 'No phone service' — handled separately.
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Verify — should now only show 'Yes' and 'No'
print("Standardization complete.")
print("OnlineSecurity values:", df['OnlineSecurity'].unique())
print("MultipleLines values :", df['MultipleLines'].unique())

### ✅ Check Your Understanding

1. **Why is collapsing 3 categories to 2 a good idea here?** Think about what a model would learn from `'No'` vs `'No internet service'`.
2. The `Churn` column has just `Yes` and `No`. **Why is no standardization needed there?**
3. **Stretch.** What if a column had `'YES'`, `'yes'`, `'Yes'`, `' Yes '`? Write the chained pandas method to fix all four to `'Yes'`. (Hint: `.str.strip().str.lower().str.capitalize()`.)

---

## 6. 🚨 Outlier Detection

An **outlier** is a value far from the rest. Outliers can be:
- **Errors** (data entry typos, sensor glitches) — usually fix or drop
- **Real but extreme** (a VIP customer, a viral product) — usually keep but cap (winsorize)
- **Exactly the signal you want** (fraud detection!) — never delete

**Always think about which case you're in BEFORE you act.**

### 6a. The IQR (Interquartile Range) method

### 📚 The math (Tukey's rule)

```
Q1   = 25th percentile (25% of values are below this)
Q3   = 75th percentile
IQR  = Q3 - Q1
lower bound = Q1 - 1.5 × IQR
upper bound = Q3 + 1.5 × IQR
anything outside [lower, upper] = outlier
```

In [ ]:
# ────────────────────────────────────────────────────────────
# Define a reusable function. 'def' starts a function definition.
# This function takes a Series and an optional 'factor' (default 1.5).
# It returns three things: the outlier mask, the lower bound, the upper bound.
# ────────────────────────────────────────────────────────────
def detect_outliers_iqr(series, factor=1.5):
    """Detect outliers in a Series using the Tukey IQR rule."""
    # quantile(0.25) = the 25th percentile = Q1
    Q1 = series.quantile(0.25)
    # quantile(0.75) = the 75th percentile = Q3
    Q3 = series.quantile(0.75)
    # Interquartile range — spread of the middle 50%
    IQR = Q3 - Q1

    # The two thresholds. 'factor=1.5' is Tukey's standard.
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR

    # True/False mask — True wherever the value falls outside [lower, upper]
    # The | operator means OR. Each condition must be wrapped in parentheses.
    outliers = (series < lower) | (series > upper)

    return outliers, lower, upper

# ────────────────────────────────────────────────────────────
# Apply our function to three numeric columns.
# ────────────────────────────────────────────────────────────
for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
    outliers, lower, upper = detect_outliers_iqr(df[col])
    # outliers.sum() = how many Trues = how many outliers
    print(f"{col}: {outliers.sum()} outliers (bounds: [{lower:.2f}, {upper:.2f}])")

In [ ]:
# ────────────────────────────────────────────────────────────
# Visualize with box plots — the BEST chart for spotting outliers.
#
# Anatomy of a box plot:
#   - The BOX is the middle 50% (Q1 to Q3)
#   - The LINE inside is the median
#   - The WHISKERS extend to the typical range
#   - DOTS above/below the whiskers are outliers
#
# plt.subplots(1, 3, ...) creates a row of 3 chart panels.
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loop with 'enumerate' to get both the index (i) and the value (col).
for i, col in enumerate(['MonthlyCharges', 'TotalCharges', 'tenure']):
    # axes[i] = the i-th panel (0, 1, 2). .boxplot() draws the chart.
    # .dropna() removes NaN values before plotting (boxplot can't handle NaN).
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)

# tight_layout() prevents labels overlapping each other.
plt.tight_layout()
plt.show()

### 6b. 📝 TODO — Z-score Method

Now you implement a second outlier detection method using **Z-scores**.

### 📚 The math

```
z = (x - mean) / std
```

A z-score tells you **how many standard deviations** a value is from the mean. The standard threshold: anything with `|z| > 3` is unusually far from the mean.

### 📚 Hint

```python
def detect_outliers_zscore(series, threshold=3):
    z = (series - series.mean()) / series.std()  # Z-score per row
    return z.abs() > threshold                    # boolean mask
```

In [ ]:
# ────────────────────────────────────────────────────────────
# TODO: Implement Z-score outlier detection.
#
#  1. Write a function detect_outliers_zscore(series, threshold=3)
#     that returns a boolean mask of outliers.
#  2. Apply it to 'MonthlyCharges', 'TotalCharges', 'tenure'.
#  3. Print how many outliers each column has.
#  4. Compare the counts to the IQR method above.
# ────────────────────────────────────────────────────────────

def detect_outliers_zscore(series, threshold=3):
    """Detect outliers using the Z-score method."""
    # YOUR CODE HERE — compute Z-scores and return |z| > threshold
    pass

# ────────────────────────────────────────────────────────────
# YOUR CODE HERE — apply to each column and print the count of outliers
# Example pattern:
# for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
#     outliers = detect_outliers_zscore(df[col].dropna())
#     print(f"{col}: {outliers.sum()} Z-score outliers")
# ────────────────────────────────────────────────────────────

### ✅ Check Your Understanding

1. Compare your Z-score outlier counts to the IQR counts. Are they similar or different? **Why?**
2. Look at the box plots. Which column has the **most extreme** outliers?
3. **Strategy question.** For `tenure` (months as a customer), an "outlier" of 72 months is just a long-term loyal customer. Should you cap it? **Why or why not?**

---

## 7. 💾 Save the Cleaned Data

Final audit, then write the cleaned DataFrame to a new CSV. **Never overwrite the raw input** — keep the original intact in case you need to start over.

In [ ]:
# ────────────────────────────────────────────────────────────
# Final sanity check — re-run the audit. Compare to the very first
# audit you ran in section 2. Every column should now make sense.
# ────────────────────────────────────────────────────────────
print("Shape:", df.shape)

# Total missing values across the whole DataFrame
print("\nMissing values:")
print(df.isnull().sum().sum(), "total")

# Quick dtype summary — how many columns of each type?
print("\nDtype counts:")
print(df.dtypes.value_counts())

In [ ]:
# ────────────────────────────────────────────────────────────
# .to_csv(filename, index=False) writes the DataFrame to a CSV.
# index=False = don't write pandas' row numbers as a column —
# without that flag you'd get an extra unnamed first column.
# ────────────────────────────────────────────────────────────
df.to_csv('telco_churn_cleaned.csv', index=False)
print("Saved: telco_churn_cleaned.csv")

# ────────────────────────────────────────────────────────────
# Colab download (uncomment to use). This pops a download dialog
# in your browser so you can save the file to your laptop.
# ────────────────────────────────────────────────────────────
# from google.colab import files
# files.download('telco_churn_cleaned.csv')

---
## 🏁 Summary — What You Learned

| Step | Pandas / Python tool | What it answers |
| --- | --- | --- |
| 🔍 Inspect | `df.shape`, `.info()`, `.describe()`, `.isnull().sum()` | *What am I looking at? What's broken?* |
| 🛠️ Type fix | `pd.to_numeric(col, errors='coerce')` | *Make text-numbers into real numbers* |
| 🕳️ Impute | `.fillna(.mean() / .median() / .mode()[0])` | *Fill missing with a sensible default* |
| 🚩 Track | `df['c_missing'] = df['c'].isna().astype(int)` | *Don't lose the missingness signal* |
| 🧽 Standardize | `.replace('old', 'new')` | *One spelling per category* |
| 🚨 Outliers (IQR) | `quantile(0.25)`, `quantile(0.75)`, Tukey rule | *What's unusually high/low?* |
| 🚨 Outliers (Z) | `(x - mean) / std`, `\|z\| > 3` | *Same idea, normality-based* |
| 💾 Save | `df.to_csv('out.csv', index=False)` | *Persist the cleaned data* |

## 🎓 Vocab Recap

- **DataFrame** — a 2D table.
- **Series** — a single column.
- **NaN** — "Not a Number" — pandas' missing-value marker.
- **dtype** — the data type of a column (`int64`, `float64`, `object`).
- **Imputation** — filling missing values with a guess.
- **Mode** — the most common value (use for categorical).
- **IQR** — Interquartile Range = Q3 − Q1.
- **Z-score** — number of standard deviations from the mean.
- **`pd.to_numeric(..., errors='coerce')`** — safe text-to-number conversion.
- **`.copy()`** — make an independent DataFrame copy (avoids hidden-share bugs).
- **`.fillna(value)`** — replace NaNs with `value`.
- **`.replace(old, new)`** — replace `old` with `new` element-wise.

## 📤 Submit

Save this notebook as `Module3_Class1_<YourName>.ipynb`. Submit a PR to your group repo at `module-3/class_1/submissions/<YourName>/`. Include `telco_churn_cleaned.csv` if you saved it.

🧹 *See you in Class 2 — Encoding and Scaling.*